# Setup

In [ ]:
# base
import os
import sys
import gc
import re
import warnings
import logging
import pickle
from time import ctime, time
from datetime import timedelta
from collections import Counter
import itertools

# data manipulation
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rpy2.robjects.conversion import localconverter

# single cell
import anndata as ad
import scanpy as sc

# custom
sys.path.insert(0, '../..')
from single_cell.R import *
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from utils import *

warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", pd.errors.DtypeWarning)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
mlogger = logging.getLogger("matplotlib")
mlogger.setLevel(logging.WARNING)

CORES = 10
DATADIR = Path("../../../data") / "processed" / "single_cell"
REFDIR = Path("../../../references")
MAIN_DIR = DATADIR / "combined"
SUBSETS_DIR = MAIN_DIR / "subsets"

METADATA = [
    "Diet",
    "Age",
    "Depot",
    "Sex",
]
DOUBLETMETHODS = ["scDblFinder", "DoubletFinder", "doubletdetection", "scrublet"]
GLOBAL_INT_KEY = "global_INT_scvi-hvg-Identifier"

converter = get_converter()
%load_ext rpy2.ipython
%matplotlib inline
# R_preload()
mpl.rcdefaults()
gc.collect()

In [ ]:
CELLTYPE_ADDON = "fibro"
CLUSTER_KEY = "leiden_fibro"
DE_KEY = "fibro_DEGs"
SELECT_RES = [0.4, 0.6]
INT_KEY = f"{CELLTYPE_ADDON}_INT_harmony-hvg-Identifier"
contam_resolutions = np.arange(1, 6) / 5
resolutions = np.arange(1, 10) / 10

# Load Data

In [ ]:
# adata = sc.read_h5ad(MAIN_DIR / "eWAT_Male.h5ad")
adata = ad.read_zarr(MAIN_DIR / "eWAT_Male.zarr")
adata

In [ ]:
BLACKLIST_GENES = adata.var_names[adata.var_names.str.contains("^ENMUSG|^Gm[0-9]|Rik", regex=True)]
BLACKLIST_GENES.to_frame().reset_index(drop=True).to_csv(REFDIR / "unknown_genes.csv")

# Fibroblasts

In [ ]:
# subset
adata_fibro = adata[adata.obs["celltype"] == "Fibroblast"].copy()
del adata_fibro.uns, adata_fibro.varm, adata_fibro.obsp
Filter_QC(adata_fibro)
Visualize(adata_fibro, key=f"{CELLTYPE_ADDON}_global", obsm=GLOBAL_INT_KEY)

### Clean contaminants

In [ ]:
# cluster
cluster = True

if cluster is True:
    Cluster(
        adata_fibro, CLUSTER_KEY, contam_resolutions, neighbor_key="neighbors_fibro"
    )

# plot clusters
for embedding in [f"UMAP_{CELLTYPE_ADDON}_global", f"LocalMAP_{CELLTYPE_ADDON}_global"]:
    r, c = 2, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(contam_resolutions):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_fibro.obs[res_key].cat.categories)
        adata_fibro.obs[res_key] = (
            adata_fibro.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_fibro,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_fibro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

In [ ]:
plot_cluster_trees(
    adata_fibro,
    [f"{CLUSTER_KEY}_{res:.1f}" for res in contam_resolutions],
    threshold=0.01,
    x_spacing=4,
    y_spacing=0.5,
)

plot_cluster_trees(
    adata_fibro,
    [f"{CLUSTER_KEY}_{res}" for res in contam_resolutions],
    markers=["Adgre1"],
    title="likely Macrophage",
    threshold=0.01,
    x_spacing=4,
    y_spacing=0.8,
)

plot_cluster_trees(
    adata_fibro,
    [f"{CLUSTER_KEY}_{res}" for res in contam_resolutions],
    markers=["Adipoq"],
    title="likely Adipocyte",
    threshold=0.01,
    x_spacing=4,
    y_spacing=0.8,
)

In [ ]:
celltype_markers = {
    "Adipocyte": ["Adipoq"],
    "Fibroblast": ["Pdgfra"],
    "Mesothlial": ["Wt1"],
    "Endothelial": ["Cdh5"],
    "Smooth Muscle Cell": ["Myocd"],
    "Pericyte": ["Enpep"],
    "Epithelial 1": ["Dcdc2a"],
    "Epithelial 2": ["Erbb4"],
    "Macrophage": ["Adgre1"],
    "Dendritic Cell": ["Flt3"],
    "Mast Cell": ["Cpa3"],
    "Neutrophil": ["Csf3r"],
    "B Cell": ["Ms4a1"],
    "T Cell": ["Cd3d"],
    "NK Cell": ["Klrd1"],
}

markers_list = pd.Series([b for a in celltype_markers for b in celltype_markers[a]])
markers_list[~markers_list.isin(adata_fibro.var_names)]

for res in [0.6]:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    clear_uns(adata_fibro, "colors")
    plot_violinplot(adata_fibro, res_key, celltype_markers, bracket_fontsize=8)

In [ ]:
# remove likely doublets
adata_fibro = adata_fibro[
    ~adata_fibro.obs[f"{CLUSTER_KEY}_0.6"].isin([6, 7, 10, 11])
].copy()
print(adata_fibro.shape)

### Try reintegration

In [ ]:
adata_fibro.X = adata_fibro.layers["normalized"].copy()
FindVariableGenes(adata_fibro, "seurat_v3", n_features=2000)

START = stopwatch(mode=2)

for integration_method in ["harmony"]:
    for pca_type in ["all", "hvg"]:
        for batch_column in ["Dataset", "Identifier"]:
            task = f"Integration Method `{integration_method}` with `{pca_type}` genes over label `{batch_column}`",
            task_start = stopwatch(task, START, mode=0)

            # Sort by batch to be contiguous (scanorama is picky or whateva)
            idx = adata_fibro.obs.sort_values(batch_column).index
            adata_fibro = adata_fibro[idx]

            # Run integration
            pca_key = f"global_PCA-{pca_type}"
            int_key = f"{CELLTYPE_ADDON}_INT_{integration_method}-{pca_type}-{batch_column}"
            Integrate(
                adata_fibro,
                batch_column,
                pca_key=pca_key,
                kind=integration_method,
                integration_key=int_key,
            )
            Visualize(adata_fibro, key=int_key, obsm=int_key, localmap=True, show=False)
            stopwatch(task, task_start, mode=1)

    adata_fibro.write_zarr(SUBSETS_DIR / f"{CELLTYPE_ADDON}_reintegrated.zarr")

In [ ]:
# compare all integrations
r, c = 2, 2
f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
axs = axs.flatten()
int_check = "Dataset"

for i, dr_key in enumerate(
    [
        f"UMAP_{CELLTYPE_ADDON}_INT_{method}-{modifier}-{batch}"
        for method in ["harmony"]
        for batch in ["Dataset", "Identifier"]
        for modifier in ["hvg", "all"]
    ]
):
    sc.pl.embedding(
        adata_fibro,
        basis=dr_key,
        color=int_check,
        alpha=1,
        ax=axs[i],
        show=False,
        legend_loc="none" if i < len(axs) - 1 else "right margin",
        palette=create_palette(8)
    )

    if i == len(axs) - 1:
        legend_info = axs[-1].get_legend_handles_labels()
        f, ax = plt.subplots()
        ax.legend(*legend_info)
        ax.axis("off")

        axs[-1].get_legend().remove()

In [ ]:
method, modifier, batch = "harmony", "hvg", "Dataset"
int_key = f"{CELLTYPE_ADDON}_INT_{method}-{modifier}-{batch}"

check_integration(
    adata_fibro, "Dataset",
    embeddings=[f"UMAP_{int_key}", f"LocalMAP_{int_key}"],
    f=plt.figure(figsize=(18,10), layout="constrained"),
    nrow=3)

method, modifier, batch = "harmony", "hvg", "Identifier"
int_key = f"{CELLTYPE_ADDON}_INT_{method}-{modifier}-{batch}"

check_integration(
    adata_fibro, "Dataset",
    embeddings=[f"UMAP_{int_key}", f"LocalMAP_{int_key}"],
    f=plt.figure(figsize=(18,10), layout="constrained"),
    nrow=3)

### Clean contaminants round 2

In [ ]:
# recluster
cluster = False

if cluster is True:
    Cluster(adata_fibro, CLUSTER_KEY, contam_resolutions, neighbor_key="neighbors_{INT_KEY}")

# plot clusters
for embedding in [f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"]:
    r, c = 3, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(tqdm(contam_resolutions)):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_fibro.obs[res_key].cat.categories)
        adata_fibro.obs[res_key] = (
            adata_fibro.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_fibro,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_fibro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

In [ ]:
celltype_markers = {
    "Adipocyte": ["Adipoq"],
    "Fibroblast": ["Pdgfra"],
    "Mesothlial": ["Wt1"],
    "Endothelial": ["Cdh5"],
    "Smooth Muscle Cell": ["Myocd"],
    "Pericyte": ["Enpep"],
    "Epithelial 1": ["Dcdc2a"],
    "Epithelial 2": ["Erbb4"],
    "Macrophage": ["Adgre1"],
    "Dendritic Cell": ["Flt3"],
    "Mast Cell": ["Cpa3"],
    "Neutrophil": ["Csf3r"],
    "B Cell": ["Ms4a1"],
    "T Cell": ["Cd3d"],
    "NK Cell": ["Klrd1"],
}

markers_list = pd.Series([b for a in celltype_markers for b in celltype_markers[a]])
markers_list[~markers_list.isin(adata_fibro.var_names)]

for res in [0.7]:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    clear_uns(adata_fibro, "colors")
    plot_violinplot(adata_fibro, res_key, celltype_markers, bracket_fontsize=8)

In [ ]:
f = plot_cluster_trees(
    adata_fibro,
    [f"{CLUSTER_KEY}_{res}" for res in contam_resolutions],
    markers=["Adgre1"],
    title="likely Macrophage",
    threshold=0.02,
    x_spacing=2,
    y_spacing=1,
)

In [ ]:
# doublet based on Adgre1 & Pdgfra/Wt1 expression
adata_fibro = adata_fibro[~adata_fibro.obs["leiden_fibro_0.3"].isin([6])]

##### Spermatid aside

In [ ]:
# module_score(adata_fibro, genesets={"germ cell" : ["Prm1", "Prm2", "Spz1", "Misfa", "Tcfl5", "Ddx4"]}, zscore=False)
# plot_violinplot(adata, f"{CLUSTER_KEY}_0.5", ["germ cell_Seurat"])

# module_score(adata_fibro, genesets={"germ cell" : ["Prm1", "Prm2", "Spz1", "Misfa", "Tcfl5", "Ddx4"]}, zscore=False)
# plot_violinplot(adata, f"celltype", ["germ cell_Seurat"])

for genes in ["germ cell_Seurat", "Adgre1"]:
    f, ax = plt.subplots(1, 1, figsize=(15, 12))
    sc.pl.embedding(
        adata_fibro,
        # basis="global_UMAP_INT_scvi-hvg-Identifier",
        basis=f"UMAP_{INT_KEY}",
        color=genes,
        ax=ax,
        show=False,
        legend_loc="on data",
        legend_fontoutline=1.5,
        legend_fontsize=20,
        palette=cluster_c,
        cmap="Reds",
        size=15,
        vmax=1,
        vmin=0.5,
    )
    ax.annotate(
        f"n = {adata_fibro.shape[0]}",
        size=15,
        fontweight="bold",
        xy=(0.98, 0.02),
        xycoords="axes fraction",
        horizontalalignment="right",
        verticalalignment="bottom",
        )

### Clustering

In [ ]:
# recluster
cluster = True

if cluster is True:
    Cluster(adata_fibro, CLUSTER_KEY, resolutions, neighbor_key=f"neighbors_{INT_KEY}")

# plot clusters
for embedding in [f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"]:
    r, c = 3, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(tqdm(resolutions)):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_fibro.obs[res_key].cat.categories)
        adata_fibro.obs[res_key] = (
            adata_fibro.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_fibro,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_fibro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

f = plot_cluster_trees(
    adata_fibro,
    [f"{CLUSTER_KEY}_{res}" for res in resolutions],
    threshold=0.03,
    node_size_range=(100, 2000),
    x_spacing=2,
    y_spacing=2,
    show_fraction=True,
)

In [ ]:
# metadata checks
embedding = f"UMAP_{INT_KEY}"

f = plt.figure(figsize=(9, 12), layout="constrained")
check_integration(
    adata_fibro,
    "Dataset",
    f,
    embeddings=[embedding],
    nrow=3,
)

f = plt.figure(figsize=(7, 10), layout="constrained")
check_integration(
    adata_fibro,
    "Diet",
    f,
    embeddings=[embedding],
    nrow=2,
)

f = plt.figure(figsize=(9, 15), layout="constrained")
check_integration(
    adata_fibro,
    "Age",
    f,
    embeddings=[embedding],
    nrow=4,
)

In [ ]:
# counts/breakdowns
for col in adata_fibro.obs.columns:
    if CLUSTER_KEY in col:
        adata_fibro.obs[col] = adata_fibro.obs[col].astype(int).astype("category")

r, c = 3, 3

# count barplots
f, axs = plt.subplots(r, c, figsize=(10 * c, 5 * r), layout="constrained")
axs = axs.flatten()
for i, res in enumerate(resolutions):
    res_key = f"{CLUSTER_KEY}_{res}"
    plot_cluster_counts(adata_fibro, res_key, ax=axs[i])

# percentage breakdowns
for col in ["Dataset", "Diet"]:
    f, axs = plt.subplots(r, c, figsize=(10 * c, 6 * r), layout="constrained")
    axs = axs.flatten()
    for i, res in enumerate(resolutions):
        res_key = f"{CLUSTER_KEY}_{res}"
        plot_cluster_stackedbarplot(adata_fibro, res_key, col, pct=True, ax=axs[i])
    f.suptitle(col + " Split", size=30)

# Exploration

### Markers

In [ ]:
# counts/breakdowns
for col in adata_fibro.obs.columns:
    if CLUSTER_KEY in col:
        adata_fibro.obs[col] = adata_fibro.obs[col].astype(int).astype("category")

r, c = 1, 3
# count barplots
f, axs = plt.subplots(r, c, figsize=(10 * c, 5 * r), layout="constrained")
axs = axs.flatten()
for i, res in enumerate(SELECT_RES):
    res_key = f"{CLUSTER_KEY}_{res}"
    plot_cluster_counts(adata_fibro, res_key, ax=axs[i])

# percentage breakdowns
for col in ["Dataset", "Diet"]:
    f, axs = plt.subplots(r, c, figsize=(10 * c, 6 * r), layout="constrained")
    axs = axs.flatten()
    for i, res in enumerate(SELECT_RES):
        res_key = f"{CLUSTER_KEY}_{res}"
        plot_cluster_stackedbarplot(adata_fibro, res_key, col, pct=True, ax=axs[i])
    f.suptitle(col + " Split", size=30)

In [ ]:
f = clustree(
    adata_fibro,
    [f"{CLUSTER_KEY}_{res}" for res in SELECT_RES],
    edge_weight_threshold=0.03,
    x_spacing=5,
    y_spacing=0.5,
    node_size_range=(100, 3000),
    show_fraction=True
)
f.set_figheight(5)

for embedding in [f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"]:
    for res in tqdm(SELECT_RES):
        res_key = f"{CLUSTER_KEY}_{res:.1f}"
        cluster_c = color_gen(adata_fibro.obs[res_key].cat.categories)
        adata_fibro.obs[res_key] = adata_fibro.obs[res_key].astype(int).astype("category")

        f, ax = plt.subplots(1, 1, figsize=(15, 12))
        sc.pl.embedding(
            adata_fibro,
            basis=embedding,
            color=res_key,
            ax=ax,
            show=False,
            legend_loc="on data",
            legend_fontoutline=1.5,
            legend_fontsize=20,
            palette=cluster_c,
            size=15,
        )
        ax.annotate(
            f"n = {adata_fibro.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

In [ ]:
# fmt: off
fb_markers = {
    "Universal FB markers": ["Pdgfra", "Dcn", "C3", "Col1a1", "Lpar1", "Ly6a", "Tcf4"],
    "Progenitors": ["Cd55", "Pi16", "Dpp4", "Dlk1", "Islr", "Fn1", "Itga5"],
    "Myofibroblasts": ["Acta2", "Cthrc1"],
    "Adipose-committed": ["Cd24a", "Itgb1", "Icam1", "Apoe", "Fabp4"],
    "Reticular": ["Thy1", "Wt1", "Fap", "Acta2", "Cnn1", "Tgm2", "Mgp", "Bst1", "Mcam"],
    "Angiogenic": ["Cxcl12", "Vegfa", "Vegfd", "Timp1"],
    "Egfr ligands": ["Egf", "Tgfa", "Hbegf", "Areg", "Btc", "Ereg", "Epgn"],
    "Dividing genes": ["Mki67", "Top2a", "Ube2c"],
    "Complement genes": ["C2", "C7", "C4a", "C4b", "Atg7"],
    "TGFb-resp low": ["Smad2", "Smad3", "Smad4", "Serpine1", "Id1", "Id2", "Id3"],
    "TGFb-resp high": ["Fn1", "Postn", "Cthrc1", "Vim", "Serpinh1"],
    "IFN genes": ["Fasn", "Irs1", "Insr", "Irf3", "Irf4", "Ifi47", "Irf1", "Ifitm1", "Ifitm2", "Ifitm3", "Isg15", "Cxcl9", "Cxcl10"],
    "General IFN/TGFb": ["Ifngr1", "Ifnar1", "Tgfbr1", "Tgfbr2", "Il33", "Il31ra", "Il17ra", "Il17rb"],
    "Collagens" : ["Col1a1", "Col1a2", "Col3a1", "Col6a1", "Col6a2", "Col6a3", "Col15a1", "Col18a1"],
    "TFs" : ["Sox9", "Zfp423", "Cebpa", "Pparg"],
    "Ungrouped": ["Saa3", "Lcn2", "Npnt", "Ces1d", "Cxcl13", "Cxcl12", "Ccl2", "Slc2a4"],
}
unique_markers = pd.Series(list(set([i for j in fb_markers.values() for i in j])))
assert np.all(unique_markers.isin(adata_fibro.var_names))

fb_paper_markers = {
    "Emont genes": ["Pde11a", "Aldh1a3", "Mgp", "Tenm3", "Epha3", "Frem1"],
    "Merrick genes": ["Dpp4", "Icam1", "F3", "Wnt6", "Thbs4", "Egfl6"],
    "Burl genes": ["Icam1", "Dpp4", "Lipe", "Top2a"],
    "Hepler genes": ["Fabp4", "Cd36", "Ly6c1"],
    "Sarvari genes": ["Foxp2", "Cd36", "Ebf2", "Klf4"],
    "Wang genes": ["Apoe", "Lifr", "Cd55", "Mfap4"],
    "Kohda genes": ["Osmr"],
}
unique_markers = pd.Series(list(set([i for j in fb_paper_markers.values() for i in j])))
assert np.all(unique_markers.isin(adata_fibro.var_names))

print(unique_markers[~unique_markers.isin(adata_fibro.var_names)].tolist())
print((~unique_markers.isin(adata_fibro.var_names)).sum())

# fmt: on

In [ ]:
for col in adata_fibro.obs.columns:
    if CLUSTER_KEY in col:
        adata_fibro.obs[col] = adata_fibro.obs[col].astype(str).astype("category")
        # adata_fibro.obs[col] = adata_fibro.obs[col].astype(int).astype("category")

for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    de_key = f"{DE_KEY}_{res:.1f}"
    # sc.pl.rank_genes_groups_stacked_violin(
    #     adata_fibro,
    #     groupby=res_key,
    #     key=de_key,
    #     var_names=fb_markers | fb_paper_markers,
    #     # values_to_plot="logfoldchanges",
    #     # cmap="bwr",
    #     # colorbar_title="log fold change",
    #     dendrogram=False,
    #     title=f"{res_key}_DEGs",
    #     vmin=-3,
    #     vmax=3,
    # )

    plot_violinplot(adata_fibro, res_key, fb_markers | fb_paper_markers)

In [ ]:
for col in adata_fibro.obs.columns:
    if CLUSTER_KEY in col:
        adata_fibro.obs[col] = adata_fibro.obs[col].astype(int).astype("category")

for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    plot_cluster_violinplot(
        adata_fibro, "Diet", res_key, ["Pdgfra"] + fb_markers["General IFN/TGFb"]
    )

### DEGs

In [ ]:
# original subclusters
RUN_DEG = True

# change to string for DEGs
for col in adata_fibro.obs.columns:
    if CLUSTER_KEY in col:
        adata_fibro.obs[col] = adata_fibro.obs[col].astype(str).astype("category")

for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    de_key = f"{DE_KEY}_{res:.1f}"
    n_clust = len(adata_fibro.obs[res_key].unique())

    # calculate DEGs
    if RUN_DEG is True:
        clear_uns(adata_fibro, res_key)
        sc.tl.rank_genes_groups(
            adata_fibro,
            groupby=res_key,
            key_added=de_key,
            use_raw=False,
            layer="normalized",
            method="wilcoxon",
        )

    # plot DEGs
    top_genes = {}
    for group, df in sc.get.rank_genes_groups_df(
        adata_fibro, group=None, key=de_key
    ).groupby("group"):
        top_genes[group] = df["names"][:30].tolist()
    f, ax = plt.subplots(1, 1, figsize=(n_clust * 5, 5), layout="constrained")
    sc.pl.rank_genes_groups_dotplot(
        adata_fibro,
        groupby=res_key,
        key=de_key,
        # n_genes=20,
        var_names=top_genes,
        # standard_scale="var",
        values_to_plot="logfoldchanges",
        cmap="bwr",
        colorbar_title="log fold change",
        ax=ax,
        title=f"{res_key}_DEGs",
        vmin=-4,
        vmax=4,
    )

    sc.pl.rank_genes_groups_heatmap(
        adata_fibro,
        key=de_key,
        groupby=res_key,
        layer="normalized",
        n_genes=50,
    )
    sc.pl.rank_genes_groups(
        adata_fibro, key=de_key, n_genes=30, title=f"{res_key}_DEGs"
    )

### Pseudotime

In [ ]:
res_key = f"{CLUSTER_KEY}_0.6"
adata_fibro_sub = adata_fibro[~(adata_fibro.obs[res_key] == '8')]

sc.tl.diffmap(adata_fibro_sub, neighbors_key=f"neighbors_{INT_KEY}", random_state=0)

comps_check = ['2,3', '4,5', '2,4']
sc.pl.embedding(
    adata_fibro_sub,
    basis="X_diffmap",
    components=comps_check,
    color=["Dpp4", "Pi16", "C7", "Adipoq", "leiden_fibro_0.6"],
    palette=color_gen(adata_fibro_sub.obs["leiden_fibro_0.6"].unique()),
    ncols=len(comps_check),
    cmap="gnuplot2",
)
diff_comp = "2,4"

In [ ]:
# sc.pp.neighbors(adata_fibro_sub, n_neighbors=10, use_rep="X_diffmap", key_added=f"neighbors_{CELLTYPE_ADDON}_diffmap")
sc.tl.paga(adata_fibro_sub, groups=res_key, neighbors_key=f"neighbors_{CELLTYPE_ADDON}_diffmap")
sc.pl.paga(adata_fibro_sub, color=["Dpp4", "Pi16", "Itgb1", "Fabp4", "Adipoq"])

In [ ]:
sc.pl.embedding(
    adata_fibro_sub,
    # basis=f"UMAP_{INT_KEY}",
    basis="X_diffmap",
    components=diff_comp,
    layer="normalized",
    color=["Dpp4", "Pi16", "Itgb1", "Fabp4", "Adipoq"],
    frameon=False,
    cmap="Reds"
)

sc.pl.embedding(
    adata_fibro_sub,
    basis=f"UMAP_{INT_KEY}",
    # basis="X_diffmap",
    # components=diff_comp,
    layer="normalized",
    color=["Dpp4", "Pi16", "Itgb1", "Fabp4", "Adipoq"],
    frameon=False,
    cmap="Reds"
)

# mark root
root_idx = adata_fibro_sub.obsm["X_diffmap"][:, 1].argmin()
root_cell_id = adata_fibro_sub.obs_names[root_idx]
adata_fibro_sub.uns['iroot'] = root_idx
adata_fibro_sub.uns[f"{CELLTYPE_ADDON}_psuedotime_root"] = root_cell_id

# plot root
adata_fibro_sub.obs["tmp"] = np.nan
adata_fibro_sub.obs.loc[root_cell_id,"tmp"] = "root"
sc.pl.embedding(
    adata_fibro_sub,
    basis="X_diffmap",
    components=[diff_comp],
    color=["tmp"],
    palette=["red"],
    size=50,
)
sc.pl.embedding(
    adata_fibro_sub,
    basis=f"UMAP_{INT_KEY}",
    color=["tmp"],
    palette=["red"],
    size=50
)
adata_fibro_sub.obs.drop(columns=["tmp"], inplace=True)

In [ ]:
sc.tl.dpt(adata_fibro_sub, neighbors_key=f"neighbors_{INT_KEY}", n_branchings=1)

In [ ]:
tmp = adata_fibro_sub[~(adata_fibro_sub.obs["dpt_pseudotime"] > 0.6)]
sc.pl.embedding(
    tmp,
    basis="X_diffmap",
    components=[diff_comp],
    layer="normalized",
    color=["dpt_pseudotime", res_key, "Dpp4", "Pi16", "Itgb1", "Fabp4", "Adipoq"],
    ncols=3,
    frameon=False,
    cmap="gnuplot2"
)
sc.pl.embedding(
    tmp,
    basis=f"UMAP_{INT_KEY}",
    layer="normalized",
    color=["dpt_pseudotime", res_key, "Dpp4", "Pi16", "Itgb1", "Fabp4", "Adipoq"],
    ncols=3,
    frameon=False,
    cmap="gnuplot2"
)
plot_violinplot(tmp, res_key, ["dpt_pseudotime"])
sc.pl.matrixplot(tmp, ["dpt_pseudotime"], res_key, cmap="Reds", swap_axes=True)

In [ ]:
# SCE approach

# adata_fibro.obsm["X_pca"] = adata_fibro.obsm["PCA_hvg"].copy()
# sce.tl.palantir(adata_fibro)
# pr_res = sce.tl.palantir_results(
#     adata_fibro,
#     early_cell=root_cell_id,
#     num_waypoints=500,
# )
# adata_fibro.obs["palantir_pseudotime"] = pr_res.pseudotime.copy()

# Native approach

import palantir
dm_res = palantir.utils.run_diffusion_maps(adata_fibro, pca_key="PCA_hvg")
ms_data = palantir.utils.determine_multiscale_space(adata_fibro)
imputed_X = palantir.utils.run_magic_imputation(adata_fibro)
terminal_states = pd.Series(
    ["Committed APC", "somethine else"],
    index=["", "Run5_134936662236454", ],
)
palantir.plot.highlight_cells_on_umap(ad, terminal_states)
pr_res = palantir.core.run_palantir(
    adata_fibro, root_cell_id, num_waypoints=500
)

In [ ]:
sc.pl.embedding(
    adata_fibro,
    # basis="UMAP_fibro",
    basis="X_diffmap",
    components=['3,4'],
    layer="normalized",
    color=["dpt_pseudotime", "palantir_pseudotime", "leiden_fibro_0.5", "Adipoq", "Dpp4", "Pi16", "Vegfd", "C7"],
    ncols=3,
    frameon=False,
)
sc.pl.embedding(
    adata_fibro,
    basis=f"UMAP_{CELLTYPE_ADDON}",
    # basis=f"LocalMAP_{CELLTYPE_ADDON}",
    color=["dpt_pseudotime", "palantir_pseudotime", "leiden_fibro_0.5", "Adipoq", "Dpp4", "Pi16", "Vegfd", "C7"],
    ncols=3,
    frameon=False,
)

In [ ]:
plot_violinplot(adata_fibro, "leiden_fibro_0.5", ["dpt_pseudotime", "palantir_pseudotime"])

### Specific Plots

In [ ]:
# adata_fibro.obsm['LocalMAP_fibro_global'] = adata_fibro.obsm['LocalMAP_fibro'].copy()
# adata_fibro.obsm['UMAP_fibro_global'] = adata_fibro.obsm['UMAP_fibro'].copy()

# adata_fibro.obsp['neighbors_fibro_global_connectivities'] = adata_fibro.obsp['neighbors_fibro_connectivities'].copy()
# adata_fibro.obsp['neighbors_fibro_global_distances'] = adata_fibro.obsp['neighbors_fibro_distances'].copy()

# adata_fibro.uns['neighbors_fibro']['connectivities_key'] = 'neighbors_fibro_global_connectivities'
# adata_fibro.uns['neighbors_fibro']['distances_key'] = 'neighbors_fibro_global_distances'
# adata_fibro.uns['neighbors_fibro_global'] = adata_fibro.uns['neighbors_fibro'].copy()

# del adata_fibro.obsm['UMAP_fibro'], adata_fibro.obsm['LocalMAP_fibro'], adata_fibro.uns['neighbors_fibro'], adata_fibro.obsp['neighbors_fibro_connectivities'], adata_fibro.obsp['neighbors_fibro_distances']

# Save/Load

In [ ]:
# save
clear_adata(adata_fibro, ["dendrogram", "colors"])
adata_fibro.write_zarr(SUBSETS_DIR / (CELLTYPE_ADDON + '.zarr'))

In [ ]:
# load
tmp = ad.read_zarr(SUBSETS_DIR / (CELLTYPE_ADDON + '.zarr'))
tmp

# Sandbox

In [ ]:
# pca_key = f"PCA-hvg_{CELLTYPE_ADDON}"
# FindVariableGenes(adata_fibro, "seurat_v3")
# PCA(adata_fibro, gene_mask="highly_variable", key=pca_key)
# Visualize(adata_fibro, key=CELLTYPE_ADDON, obsm=pca_key)
# Cluster(adata_fibro, CLUSTER_KEY, resolutions, neighbor_key=f"neighbors_{CELLTYPE_ADDON}")